# 21 — Ensemble dei punteggi dei 5 semi di PWfE

Nessun addestramento e nessuna nuova inferenza: si usano i punteggi salvati dal notebook 19
(`punteggi_dev/` e `punteggi_eval/`, file `PWfE_{200,150}_s{0..4}.npy`).

**Ensemble** = per ogni file, media dei logit dei modelli. Due varianti, fissate prima di guardare l'eval:
- `logit`: media dei logit grezzi;
- `z_dev`: ogni modello viene prima standardizzato con media e deviazione standard dei **suoi punteggi sul dev 2019**
  (le scale dei logit differiscono tra semi), poi si fa la media.

Gruppi: `ENS_200` (5 semi senza sovrapposizione), `ENS_150` (5 semi con sovrapposizione), `ENS_tutti` (10 modelli).

**Regola fissata a priori:** per ogni gruppo la variante preferita è quella con EER **dev** più basso; l'eval non entra
in nessuna scelta; a parità vince `logit`, la più semplice. Si riportano comunque entrambe.

In [1]:
import os, json, hashlib, importlib.util, urllib.request
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support, confusion_matrix)

INPUT_ROOT = Path(os.environ.get("INPUT_ROOT", "/kaggle/input"))
BASE = INPUT_ROOT / "datasets/angelopaldino"
DF_KEYS = Path(os.environ.get("DF_KEYS", BASE / "df-keys/CM/trial_metadata.txt"))
DEV_INDEX = Path(os.environ.get("DEV_INDEX", BASE / "devroot/root/index_dev.csv"))
NB19_DIR = os.environ.get("NB19_DIR", "")      # cartella che contiene punteggi_dev/ e punteggi_eval/; vuoto = ricerca automatica
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/ensemble_pwfe")); OUT_DIR.mkdir(parents=True, exist_ok=True)
N_BOOT = int(os.environ.get("N_BOOT", 1000))
SEMI = [0, 1, 2, 3, 4]

# EER eval (%) dei singoli modelli dal notebook 19 (risultati_eval.csv): controllo di caricamento
RIF_EVAL = {"PWfE_200_s0": 16.0872, "PWfE_200_s1": 17.7223, "PWfE_200_s2": 17.4877, "PWfE_200_s3": 13.9821,
            "PWfE_200_s4": 16.5445, "PWfE_150_s0": 17.2025, "PWfE_150_s1": 14.2444, "PWfE_150_s2": 14.5471,
            "PWfE_150_s3": 17.4524, "PWfE_150_s4": 17.5130}

def trova_nb19():
    if NB19_DIR:
        return Path(NB19_DIR)
    serve = {f"{k}.npy" for k in RIF_EVAL}
    hits = [Path(d).parent for d, _, f in os.walk(INPUT_ROOT)
            if Path(d).name == "punteggi_eval" and serve <= set(f) and (Path(d).parent / "punteggi_dev").is_dir()]
    assert len(hits) == 1, f"cartelle del notebook 19 trovate: {hits}; imposta NB19_DIR"
    return hits[0]
D19 = trova_nb19(); print("notebook 19:", D19)

EVAL_METRICS_URL = "https://raw.githubusercontent.com/asvspoof-challenge/2021/main/eval-package/eval_metrics.py"
p = OUT_DIR / "eval_metrics.py"
if not p.exists():
    urllib.request.urlretrieve(EVAL_METRICS_URL, p)
SHA = hashlib.sha256(p.read_bytes()).hexdigest(); assert SHA.startswith("70c5d2fe8fab6654"), SHA[:16]
spec = importlib.util.spec_from_file_location("asv_metrics", p); asv = importlib.util.module_from_spec(spec); spec.loader.exec_module(asv)

def eer_soglia(y, s):
    s = np.asarray(s, np.float64); eer, thr = asv.compute_eer(s[y == 1], s[y == 0]); return float(eer), float(thr)

notebook 19: /kaggle/input/datasets/angelo01paldino/pwfe-new/PWfE_riaddestrato_inferenza/pwfe_riaddestrato


In [2]:
COLS = ["speaker", "utt_id", "codec", "source", "attack", "label", "trim", "subset", "vocoder", "c9", "c10", "c11", "c12"]
meta = pd.read_csv(DF_KEYS, sep=r"\s+", header=None, names=COLS, dtype=str, engine="python")
meta = meta[meta["subset"] == "eval"].reset_index(drop=True)
Y_E = (meta["label"] == "bonafide").to_numpy().astype(int)
Y_D = pd.read_csv(DEV_INDEX)["y_bonafide"].to_numpy().astype(int)
print(f"eval: {len(Y_E)} prove ({Y_E.sum()} bona fide) | dev: {len(Y_D)} prove ({Y_D.sum()} bona fide)")

SD, SE = {}, {}
for k, rif in RIF_EVAL.items():
    SD[k] = np.load(D19 / "punteggi_dev" / f"{k}.npy").astype(np.float64)
    SE[k] = np.load(D19 / "punteggi_eval" / f"{k}.npy").astype(np.float64)
    assert len(SD[k]) == len(Y_D) and len(SE[k]) == len(Y_E), k
    e = 100 * eer_soglia(Y_E, SE[k])[0]
    assert abs(e - rif) < 1e-3, f"{k}: EER eval {e:.4f} invece di {rif}"
print("controllo superato: i 10 modelli riproducono gli EER eval del notebook 19")

GRUPPI = {"ENS_200": [f"PWfE_200_s{s}" for s in SEMI], "ENS_150": [f"PWfE_150_s{s}" for s in SEMI]}
GRUPPI["ENS_tutti"] = GRUPPI["ENS_200"] + GRUPPI["ENS_150"]

def ensemble(nomi, variante, split):
    S = SD if split == "dev" else SE
    if variante == "logit":
        return np.mean([S[n] for n in nomi], axis=0)
    # z_dev: statistiche sempre dal dev, applicate a dev ed eval
    return np.mean([(S[n] - SD[n].mean()) / SD[n].std() for n in nomi], axis=0)

ENS_D, ENS_E = {}, {}
for g, nomi in GRUPPI.items():
    for v in ("logit", "z_dev"):
        ENS_D[f"{g}_{v}"] = ensemble(nomi, v, "dev"); ENS_E[f"{g}_{v}"] = ensemble(nomi, v, "eval")

eval: 533928 prove (14869 bona fide) | dev: 24844 prove (2548 bona fide)
controllo superato: i 10 modelli riproducono gli EER eval del notebook 19


In [3]:
def metriche(y, s):
    eer, thr = eer_soglia(y, s)
    pred = (s > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    P, R, F, _ = precision_recall_fscore_support(y, pred, labels=[1, 0], zero_division=0)
    return {"EER_%": 100 * eer, "AUC": roc_auc_score(y, s), "soglia_eer": thr, "FP": int(fp), "FN": int(fn),
            "FPR_%": 100 * fp / (fp + tn), "FNR_%": 100 * fn / (fn + tp),
            "accuratezza_%": 100 * accuracy_score(y, pred), "accuratezza_bilanciata_%": 100 * balanced_accuracy_score(y, pred),
            "precision_bonafide_%": 100 * P[0], "recall_bonafide_%": 100 * R[0], "F1_bonafide_%": 100 * F[0],
            "precision_spoof_%": 100 * P[1], "recall_spoof_%": 100 * R[1], "F1_spoof_%": 100 * F[1],
            "F1_macro_%": 100 * F.mean()}

righe = []
for k in RIF_EVAL:
    righe.append({"modello": k, "tipo": "singolo", "EER_dev_%": 100 * eer_soglia(Y_D, SD[k])[0], **metriche(Y_E, SE[k])})
for k in ENS_E:
    righe.append({"modello": k, "tipo": "ensemble", "EER_dev_%": 100 * eer_soglia(Y_D, ENS_D[k])[0], **metriche(Y_E, ENS_E[k])})
RIS = pd.DataFrame(righe)
RIS.to_csv(OUT_DIR / "metriche_ensemble.csv", index=False)

# variante preferita per gruppo: EER dev piu' basso (regola a priori)
PREF = {}
for g in GRUPPI:
    sub = RIS[RIS["modello"].isin([f"{g}_logit", f"{g}_z_dev"])]
    PREF[g] = sub.loc[sub["EER_dev_%"].idxmin(), "modello"]
print("variante preferita sul dev:", PREF)

sing = RIS[RIS["tipo"] == "singolo"].assign(config=lambda d: d["modello"].str.rsplit("_s", n=1).str[0])
media_semi = sing.groupby("config")[["EER_dev_%", "EER_%", "AUC"]].agg(["mean", "std"])
pd.set_option("display.width", 250)
print("\n=== singoli modelli: media sui semi ==="); display(media_semi.round(3))
print("\n=== ensemble ==="); display(RIS[RIS["tipo"] == "ensemble"].set_index("modello")
        [["EER_dev_%", "EER_%", "AUC", "FPR_%", "FNR_%", "accuratezza_%", "accuratezza_bilanciata_%", "F1_macro_%"]].round(3))

variante preferita sul dev: {'ENS_200': 'ENS_200_z_dev', 'ENS_150': 'ENS_150_z_dev', 'ENS_tutti': 'ENS_tutti_z_dev'}

=== singoli modelli: media sui semi ===


EER_dev_%          EER_%           AUC       
              mean    std    mean    std   mean    std
config                                                
PWfE_150     1.309  0.196  16.192  1.647  0.917  0.014
PWfE_200     1.327  0.033  16.365  1.491  0.913  0.012


=== ensemble ===


,EER_dev_%,EER_%,AUC,FPR_%,FNR_%,accuratezza_%,accuratezza_bilanciata_%,F1_macro_%
modello,,,,,,,,
ENS_200_logit,0.856,15.810,0.924,15.808,15.811,84.192,84.190,57.035
ENS_200_z_dev,0.822,15.859,0.923,15.859,15.858,84.141,84.141,56.987
ENS_150_logit,0.825,15.051,0.926,15.051,15.051,84.949,84.949,57.782
ENS_150_z_dev,0.745,14.788,0.928,14.787,14.789,85.213,85.212,58.052
ENS_tutti_logit,0.785,15.138,0.927,15.136,15.139,84.864,84.862,57.696
ENS_tutti_z_dev,0.745,14.977,0.928,14.977,14.977,85.023,85.023,57.857


In [4]:
def bootstrap_appaiato(y, s_a, s_b, n_boot=N_BOOT, seed=0):
    g = np.random.default_rng(seed)
    ib, is_ = np.where(y == 1)[0], np.where(y == 0)[0]
    d = []
    for _ in range(n_boot):
        ii = np.concatenate([g.choice(ib, len(ib)), g.choice(is_, len(is_))])
        d.append(eer_soglia(y[ii], s_a[ii])[0] - eer_soglia(y[ii], s_b[ii])[0])
    return np.percentile(100 * np.array(d), [2.5, 97.5])

# contrasti (eval): ensemble preferito contro ciascun singolo seme dello stesso gruppo, e ENS_150 contro ENS_200
contr = []
for g in ("ENS_200", "ENS_150"):
    for n in GRUPPI[g]:
        lo, hi = bootstrap_appaiato(Y_E, ENS_E[PREF[g]], SE[n])
        contr.append({"confronto": f"{PREF[g]} - {n}",
                      "delta_%": 100 * (eer_soglia(Y_E, ENS_E[PREF[g]])[0] - eer_soglia(Y_E, SE[n])[0]), "ic_lo": lo, "ic_hi": hi})
lo, hi = bootstrap_appaiato(Y_E, ENS_E[PREF["ENS_150"]], ENS_E[PREF["ENS_200"]])
contr.append({"confronto": f"{PREF['ENS_150']} - {PREF['ENS_200']}",
              "delta_%": 100 * (eer_soglia(Y_E, ENS_E[PREF["ENS_150"]])[0] - eer_soglia(Y_E, ENS_E[PREF["ENS_200"]])[0]),
              "ic_lo": lo, "ic_hi": hi})
CONTR = pd.DataFrame(contr); CONTR.to_csv(OUT_DIR / "contrasti_ensemble.csv", index=False)
display(CONTR.round(3))

(OUT_DIR / "riepilogo_ensemble.json").write_text(json.dumps({
    "punteggi": str(D19), "gruppi": GRUPPI, "varianti": {"logit": "media dei logit grezzi",
        "z_dev": "media dei logit standardizzati con media e std dei punteggi dev di ciascun modello"},
    "regola": "variante preferita per gruppo = EER dev piu' basso", "preferite": PREF,
    "ensemble": RIS[RIS["tipo"] == "ensemble"].set_index("modello")[["EER_dev_%", "EER_%", "AUC"]].round(4).to_dict("index"),
    "singoli_media": {c: {"EER_eval_media": float(sing[sing.config == c]["EER_%"].mean()),
                          "EER_eval_std": float(sing[sing.config == c]["EER_%"].std())} for c in sing.config.unique()},
    "bootstrap": {"n_boot": N_BOOT, "tipo": "appaiato, stratificato per classe"},
    "eer_function": {"source": EVAL_METRICS_URL, "sha256": SHA},
    "limiti": ["soglia di EER calcolata a posteriori su eval",
               "l'ensemble richiede di eseguire 5 (o 10) teste in inferenza; gli embedding e la prosodia restano condivisi"]},
    indent=2, default=float))
print("Output:", sorted(x.name for x in OUT_DIR.iterdir()))

,confronto,delta_%,ic_lo,ic_hi
0,ENS_200_z_dev - PWfE_200_s0,-0.229,-0.496,0.047
1,ENS_200_z_dev - PWfE_200_s1,-1.864,-2.097,-1.607
2,ENS_200_z_dev - PWfE_200_s2,-1.629,-1.863,-1.379
3,ENS_200_z_dev - PWfE_200_s3,1.876,1.596,2.153
4,ENS_200_z_dev - PWfE_200_s4,-0.686,-0.929,-0.463
5,ENS_150_z_dev - PWfE_150_s0,-2.415,-2.642,-2.185
6,ENS_150_z_dev - PWfE_150_s1,0.543,0.330,0.775
7,ENS_150_z_dev - PWfE_150_s2,0.241,0.007,0.444
8,ENS_150_z_dev - PWfE_150_s3,-2.665,-2.907,-2.462
9,ENS_150_z_dev - PWfE_150_s4,-2.725,-2.979,-2.462


Output: ['__pycache__', 'contrasti_ensemble.csv', 'eval_metrics.py', 'metriche_ensemble.csv', 'riepilogo_ensemble.json']
